# Back2speaK — Analyse exploratoire & Classification binaire
**Objectif** : détecter si la prononciation du phonème 'che' est correcte ou incorrecte.  
**Dataset** : 407 enregistrements audio (368 avec fichier disponible), très déséquilibré (~94% correct).

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))  # add project root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import librosa
import librosa.display
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
%matplotlib inline
print('Imports OK')

## 1. Chargement des données

In [ ]:
from src.data_loader import load_labels, class_distribution
from src.utils import set_seed
set_seed()

df = load_labels()
print(f'Samples disponibles : {len(df)}')
df.head()

### 1.1 Distribution des classes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw decision
df['decision'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Décision (brute)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

# Binary
label_names = {0: 'Incorrecte (0)', 1: 'Correcte (1)'}
df['label'].map(label_names).value_counts().plot(kind='bar', ax=axes[1],
    color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[1].set_title('Label binaire')
axes[1].tick_params(axis='x', rotation=15)

# Age distribution by label
for lbl, grp in df.groupby('label'):
    axes[2].hist(grp['age'], bins=15, alpha=0.6, label=label_names[lbl])
axes[2].set_title('Distribution des âges par classe')
axes[2].set_xlabel('Âge')
axes[2].legend()

plt.tight_layout()
plt.savefig('../results/eda_distributions.png', dpi=150)
plt.show()

print(f"\nDéséquilibre : {df['label'].mean()*100:.1f}% correct ({df['label'].sum()}/{len(df)})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Position
pos_ct = df.groupby(['position','label']).size().unstack(fill_value=0)
pos_ct.plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[0].set_title('Position du phonème')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(['Incorrecte','Correcte'])

# Type item
type_ct = df.groupby(['type_item','label']).size().unstack(fill_value=0)
type_ct.plot(kind='bar', ax=axes[1], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[1].set_title("Type d'item")
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(['Incorrecte','Correcte'])

plt.tight_layout()
plt.show()

## 2. Visualisation audio

In [ ]:
from src.preprocessing import load_audio, TARGET_SR

# Pick one correct and one incorrect example
correct_row   = df[df['label']==1].iloc[0]
incorrect_row = df[df['label']==0].iloc[0]

y_correct   = load_audio(correct_row['audio_path'])
y_incorrect = load_audio(incorrect_row['audio_path'])

fig, axes = plt.subplots(2, 3, figsize=(16, 7))

for i, (y, label, row) in enumerate([
    (y_correct,   'Correcte',   correct_row),
    (y_incorrect, 'Incorrecte', incorrect_row),
]):
    # Waveform
    times = np.linspace(0, len(y)/TARGET_SR, len(y))
    axes[i,0].plot(times, y, lw=0.5, alpha=0.8)
    axes[i,0].set_title(f'{label} — Forme d\'onde ({row["audio_id"]})')
    axes[i,0].set_xlabel('Temps (s)')

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=40)
    img  = librosa.display.specshow(mfcc, sr=TARGET_SR, x_axis='time', ax=axes[i,1])
    axes[i,1].set_title(f'{label} — MFCC')
    fig.colorbar(img, ax=axes[i,1])

    # Log-Mel spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img2 = librosa.display.specshow(mel_db, sr=TARGET_SR, x_axis='time',
                                     y_axis='mel', ax=axes[i,2])
    axes[i,2].set_title(f'{label} — Mel-spectrogram')
    fig.colorbar(img2, ax=axes[i,2], format='%+2.0f dB')

plt.tight_layout()
plt.savefig('../results/audio_visualisations.png', dpi=150)
plt.show()

## 3. Extraction de features

In [ ]:
from src.features import build_feature_matrix, extract_audio_features

# Load all waveforms
print('Chargement des fichiers audio…')
waveforms = [load_audio(row['audio_path']) for _, row in df.iterrows()]

print('Extraction des features…')
X = build_feature_matrix(df, waveforms, use_metadata=True)
y = df['label'].values

print(f'Matrice de features : {X.shape}  — {X.shape[1]} features par sample')
print(f'Labels : {np.bincount(y)}')

In [ ]:
# Feature statistics
print('NaN:', np.isnan(X).sum())
print('Inf:', np.isinf(X).sum())

# t-SNE visualisation (réduire à 2D)
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

X_scaled = StandardScaler().fit_transform(X)
X_2d = TSNE(n_components=2, random_state=42, perplexity=15).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 5))
colors = {1: '#2ecc71', 0: '#e74c3c'}
labels = {1: 'Correcte', 0: 'Incorrecte'}
for lbl in [0, 1]:
    mask = y == lbl
    ax.scatter(X_2d[mask,0], X_2d[mask,1], c=colors[lbl],
               label=labels[lbl], alpha=0.7, s=30)
ax.set_title('t-SNE des features audio (2D)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/tsne.png', dpi=150)
plt.show()

## 4. Entraînement des modèles

In [ ]:
from sklearn.model_selection import train_test_split
from src.train import build_pipelines, cross_validate_models
from src.utils import SEED

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
print(f'Train: {len(y_train)} | Test: {len(y_test)}')
print(f'Train pos: {y_train.sum()} | Test pos: {y_test.sum()}')

In [ ]:
pipelines  = build_pipelines()
cv_results = cross_validate_models(X_train, y_train, pipelines)
cv_results

In [ ]:
from src.evaluate import plot_cv_results
plot_cv_results(cv_results)
plt.show()

## 5. Évaluation du meilleur modèle

In [ ]:
from src.evaluate import compute_metrics, plot_confusion_matrix, plot_roc_curve
from sklearn.metrics import classification_report

best_name = cv_results.iloc[0]['model']
print(f'Best model: {best_name}')

best_pipe = pipelines[best_name]
best_pipe.fit(X_train, y_train)

y_pred = best_pipe.predict(X_test)
y_prob = best_pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred,
      target_names=['Incorrecte', 'Correcte'], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
import seaborn as sns

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Incorrecte','Correcte'],
            yticklabels=['Incorrecte','Correcte'], ax=axes[0])
axes[0].set_title('Matrice de confusion')
axes[0].set_xlabel('Prédiction'); axes[0].set_ylabel('Vérité')

# ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_title('Courbe ROC')
axes[1].set_xlabel('Faux positifs'); axes[1].set_ylabel('Vrais positifs')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/final_evaluation.png', dpi=150)
plt.show()

In [ ]:
metrics = compute_metrics(y_test, y_pred, y_prob)
pd.Series(metrics).rename('Score').to_frame()

## 6. Analyse des erreurs

In [ ]:
# Reconstruct test DataFrame
from sklearn.model_selection import train_test_split
_, df_test = train_test_split(df, test_size=0.15, stratify=y, random_state=SEED)
df_test = df_test.copy().reset_index(drop=True)
df_test['y_pred'] = y_pred
df_test['y_prob'] = y_prob
df_test['correct_pred'] = df_test['label'] == df_test['y_pred']

# False Negatives (incorrecte prédit comme correcte) — les plus coûteux
fn = df_test[(df_test['label']==0) & (df_test['y_pred']==1)]
fp = df_test[(df_test['label']==1) & (df_test['y_pred']==0)]
print(f'Faux négatifs (manqués): {len(fn)}')
print(f'Faux positifs (surestimés): {len(fp)}')
if len(fn):
    print('\nFaux négatifs:')
    print(fn[['audio_id','age','sexe','position','type_item','decision','y_prob']].to_string())

## 7. Sauvegarde du modèle final


In [ ]:
import joblib, os
from src.utils import MODELS_DIR

save_path = os.path.join(MODELS_DIR, f'{best_name}_final.joblib')
joblib.dump(best_pipe, save_path)
print(f'Modèle sauvegardé → {save_path}')

# Quick inference test
loaded = joblib.load(save_path)
test_pred = loaded.predict(X_test[:3])
print(f'Test inference: {test_pred}  (expected: {y_test[:3]})')

## 8. Rapport de synthèse


In [ ]:
print('='*60)
print('RAPPORT DE PERFORMANCE — Back2speaK')
print('='*60)
print(f'Dataset  : {len(df)} échantillons ({df["label"].sum()} correct, {(df["label"]==0).sum()} incorrect)')
print(f'Test set : {len(y_test)} échantillons')
print(f'Modèle   : {best_name}')
print()
print('Métriques test :')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.4f}')
print()
print('Cross-validation (train, 5-fold) :')
best_cv = cv_results[cv_results['model']==best_name].iloc[0]
print(f'  F1    : {best_cv["f1_mean"]:.3f} ± {best_cv["f1_std"]:.3f}')
print(f'  AUC   : {best_cv["roc_auc_mean"]:.3f} ± {best_cv["roc_auc_std"]:.3f}')
print('='*60)